In [23]:
from dataset import BrainToTextDataset, train_test_split_indicies
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
import os
import numpy as np
import pathlib
import os
import sys
from evaluate_model_helpers import *

In [12]:
sessions = ['t15.2023.08.11',
    't15.2023.08.13',
    't15.2023.08.18',
    't15.2023.08.20',
    't15.2023.08.25',
    't15.2023.08.27',
    't15.2023.09.01',
    't15.2023.09.03',
    't15.2023.09.24',
    't15.2023.09.29',
    't15.2023.10.01',
    't15.2023.10.06',
    't15.2023.10.08',
    't15.2023.10.13',
    't15.2023.10.15',
    't15.2023.10.20',
    't15.2023.10.22',
    't15.2023.11.03',
    't15.2023.11.04',
    't15.2023.11.17',
    't15.2023.11.19',
    't15.2023.11.26',
    't15.2023.12.03',
    't15.2023.12.08',
    't15.2023.12.10',
    't15.2023.12.17',
    't15.2023.12.29',
    't15.2024.02.25',
    't15.2024.03.03',
    't15.2024.03.08',
    't15.2024.03.15',
    't15.2024.03.17',
    't15.2024.04.25',
    't15.2024.04.28',
    't15.2024.05.10',
    't15.2024.06.14',
    't15.2024.07.19',
    't15.2024.07.21',
    't15.2024.07.28',
    't15.2025.01.10',
    't15.2025.01.12',
    't15.2025.03.14',
    't15.2025.03.16',
    't15.2025.03.30',
    't15.2025.04.13'
]

In [13]:
train_file_paths = [os.path.join('../data/hdf5_data_final',s,'data_train.hdf5') for s in sessions]

In [14]:
train_trials, _ = train_test_split_indicies(
    file_paths = train_file_paths,
    test_percentage = 0,
    seed = 10,
    bad_trials_dict = None,
)

In [16]:
dataset = BrainToTextDataset(
    trial_indicies = train_trials,
    split = 'train',
    days_per_batch = 4,
    n_batches = 120000,
    batch_size = 64,
    must_include_days = None,
    random_seed = 10,
    feature_subset = None
    )

In [17]:
dataloader = DataLoader(
    dataset,
    batch_size = None,
    shuffle = False,
    num_workers = 4,
    pin_memory = True
)

In [28]:
LOGIT_TO_DIPHONE = []
for a in LOGIT_TO_PHONEME:
    for b in LOGIT_TO_PHONEME:
        LOGIT_TO_DIPHONE.append((a, b))

In [47]:
def phoneme_to_diphone(labels, phone_seq_lens):
        diphone_labels = []
        diphone_seq_lens = []
        for label, seq_len in zip(labels, phone_seq_lens):
            diphone_label = []
            for i in range(seq_len - 1):
                phone_a = label[i].item()
                phone_b = label[i+1].item()
                diphone = (LOGIT_TO_PHONEME[phone_a], LOGIT_TO_PHONEME[phone_b])
                diphone_idx = LOGIT_TO_DIPHONE.index(diphone)
                diphone_label.append(diphone_idx)
            diphone_seq_lens.append(torch.tensor(len(diphone_label)))
            if len(diphone_label) < label.numel():
                pad = torch.zeros(label.numel() - len(diphone_label), dtype=int)
                diphone_label = torch.cat((torch.tensor(diphone_label), pad))
            diphone_labels.append(diphone_label)
        return torch.stack(diphone_labels), torch.stack(diphone_seq_lens)

In [48]:
for i, batch in enumerate(dataloader):
    if i > 0:
        break
    labels = batch['seq_class_ids'].to('cpu')
    print(labels[0])
    print(labels.shape)
    phone_seq_lens = batch['phone_seq_lens'].to('cpu')
    print(phone_seq_lens[0])
    print(phone_seq_lens.shape)
    diphone_labels, diphone_seq_lens = phoneme_to_diphone(labels, phone_seq_lens)


tensor([ 6, 22, 40, 14, 28,  3, 22, 40,  3, 27, 40, 23,  4, 28, 32, 40,  4, 21,
        29, 25, 40,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0, 

In [50]:
print(diphone_labels[0])
print(diphone_labels.shape)
print(diphone_seq_lens[0])
print(diphone_seq_lens.shape)

tensor([ 268,  942, 1654,  602, 1151,  145,  942, 1643,  150, 1147, 1663,  947,
         192, 1180, 1352, 1644,  185,  890, 1214, 1065,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,   

In [35]:
LOGIT_TO_DIPHONE[:10]

[('BLANK', 'BLANK'),
 ('BLANK', 'AA'),
 ('BLANK', 'AE'),
 ('BLANK', 'AH'),
 ('BLANK', 'AO'),
 ('BLANK', 'AW'),
 ('BLANK', 'AY'),
 ('BLANK', 'B'),
 ('BLANK', 'CH'),
 ('BLANK', 'D')]

In [36]:
LOGIT_TO_DIPHONE.index(('BLANK', 'B'))

7